# Week 5 - PyMongo CRUD and Ingestion

This notebook continues the previous MongoDB class.

Important idea:
- reuse the MongoDB container from the previous class
- reuse the same database: `university_mongo`
- create a **new collection** for the external dataset
- do **not** perform EDA here
- perform only small validations to confirm the pipeline is working

## Learning goals

By the end of this notebook, you should be able to:

1. Connect to MongoDB from Python with `PyMongo`.
2. Organize connection logic inside a class.
3. Create reusable methods for select, insert, update, and delete.
4. Read an external raw CSV dataset.
5. Create a new collection and ingest data into it.
6. Run a few small checks to confirm the ingestion worked.

## Dataset choice for this practice

We will use the UCI Student Performance dataset already stored in:

- `../../raw/uci-student-performance/dataset/student-mat.csv`
- `../../raw/uci-student-performance/dataset/student-por.csv`

These files use `;` as the separator.

The new collection to create in MongoDB will be:

- `student_performance`

## Imports

Complete the imports needed for:

- file paths
- JSON handling
- pandas
- MongoDB client

In [1]:
from pathlib import Path
import json

import pandas as pd
from pymongo import MongoClient
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

## Connection settings

Use the same MongoDB service from the previous practical class.

Credentials source for this notebook:
- `prj/docker-compose.yml`
- root user: `labuser`
- root password: `labpass`
- authentication database: `admin`
- recommended host in this notebook: `host.docker.internal`

If the connection fails with `Authentication failed`, first test the container manually with:
- `docker exec -it lab-mongo mongosh -u labuser -p labpass --authenticationDatabase admin`

If that command also fails, the running volume was probably initialized earlier with different credentials.

Important local environment note:
- if you already have another local `mongod` running on port `27017`, `localhost` may connect to that service instead of the class container
- for this reason, prefer `host.docker.internal` as the MongoDB host in the notebook configuration

Database to reuse:
- `university_mongo`

New collection to create for this class:
- `student_performance`

In [2]:
# TODO: define the connection settings
# Use the credentials declared in prj/docker-compose.yml
# Prefer host.docker.internal instead of localhost to avoid a local mongod conflict
# Remember that authentication is performed against the admin database
# Example variables to create:
MONGO_CONFIG = {
    "user": "labuser",
    "password": "labpass",
    "host": "host.docker.internal",
    "port": 27017,
    "database": "university_mongo"
}

# RAW_DIR = Path(...)
# COLLECTION_NAME = ...


## Documentation references:

- MongoDB PyMongo docs: [MongoClient](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.13/connect/mongoclient/)
- MongoDB PyMongo docs: [Find](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/find/)
- MongoDB PyMongo docs: [Insert](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/insert/)
- MongoDB PyMongo docs: [Update](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.10/crud/update/)
- MongoDB PyMongo docs: [Delete](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.10/crud/delete/)

- UCI dataset page: Student Performance: https://archive.ics.uci.edu/dataset/320/student+performance

## Step 1 - Create a database class

Model your solution with the same idea used in previous database exercises:

- store configuration in the constructor
- create a `connect()` method
- create a `close()` method
- provide a method to retrieve a collection

This class should focus only on connection management.

In [3]:
class MongoDatabase:
    """Class responsible for MongoDB connection management."""

    def __init__(self, db_config):
        # TODO: store configuration fields such as host, port, user, password, database
        self.client = None
        self.db = None

    def connect(self):
        # TODO: create a MongoClient only if it does not exist yet
        # TODO: build the URI with authSource=admin
        # TODO: set a small serverSelectionTimeoutMS for friendlier failures
        # TODO: select the target database
        # TODO: test the connection with ping
        # TODO: if authentication fails, raise a clear message mentioning
        #       prj/docker-compose.yml and the docker exec mongosh command
        return None

    def is_connected(self):
        # TODO: return True if the client is available, otherwise False
        pass

    def get_collection(self, collection_name):
        # TODO: return the collection object from the selected database
        pass

    def close(self):
        # TODO: close the client and reset internal references
        pass


## Step 2 - Create a CRUD executor class

This second class should focus on collection operations, not on connection setup.

Suggested responsibilities:

- select documents
- insert one document
- insert many documents for ingestion
- update one document
- delete one document
- count documents

In [ ]:
class MongoExecutor:
    """Class responsible for CRUD operations on a MongoDB collection."""

    def __init__(self, database, collection_name):
        # TODO: keep a reference to the database object
        # TODO: obtain the collection from the database object
        pass

    def select(self, filter_query=None, projection=None, limit=10):
        # TODO: implement a simple find() wrapper
        pass

    def insert_one(self, document):
        # TODO: insert a single document
        pass

    def insert_many(self, documents):
        # TODO: insert many documents for the ingestion step
        pass

    def update_one(self, filter_query, update_query):
        # TODO: update a single document
        pass

    def delete_one(self, filter_query):
        # TODO: delete a single document
        pass

    def count_documents(self, filter_query=None):
        # TODO: count documents, defaulting to all documents
        pass


## Step 3 - Instantiate and test the connection

At this point, you should:

1. instantiate the `MongoDatabase`
2. connect to the database
3. instantiate the `MongoExecutor` for the new collection
4. confirm that the connection is working

Recommended check:
- call `db.connect()`
- if it fails, verify the same credentials with `mongosh` inside the container first

In [ ]:
# TODO: instantiate the database object
# TODO: connect to MongoDB


# TODO: instantiate the executor for student_performance
# TODO: print a small success message or a count

### Test CRUD methods with small document

In [ ]:
demo_document = {
    "external_student_id": "DEMO-00001",
    "source_dataset": "manual_demo",
    "source_file": "manual",
    "school": "GP",
    "sex": "F",
    "age": 17,
    "studytime": 2,
    "failures": 0,
    "absences": 1,
    "G1": 12,
    "G2": 13,
    "G3": 14,
    "internet": "yes",
    "higher": "yes"
}

# uncomment the following lines to test the CRUD methods with a small document
# executor.insert_one(demo_document)
# executor.update_one({"external_student_id": "DEMO-00001"}, {"$set": {"G3": 15}})
# executor.select({"external_student_id": "DEMO-00001"}, {"_id": 0}, limit=1)


In [ ]:
# delete the demo document by id after testing
# executor.delete_one({"external_student_id": "DEMO-00001"})

## Step 4 - Read the raw CSV files

Read both CSV files from `RAW_DIR`.

Remember:
- the separator is `;`
- we are still not doing EDA
- just load and inspect a few rows

In [ ]:
# TODO: create paths for student-mat.csv and student-por.csv
# TODO: load them with pandas using sep=';'
# TODO: display a small sample


## Step 5 - Map a raw row into a MongoDB document

Instead of inserting the raw CSV row blindly, create a mapping function.

Suggested output fields:

- `external_student_id`
- `source_dataset`
- `source_file`
- `school`
- `sex`
- `age`
- `studytime`
- `failures`
- `absences`
- `G1`, `G2`, `G3`
- `internet`
- `higher`


In [ ]:
def map_external_record(record, dataset_name, row_number):
    """Transform one CSV row into one MongoDB document."""

    # TODO: build a stable document structure for the new collection
    # TODO: use a generated external_student_id such as UCI-MAT-00001
    # TODO: keep the source metadata in the document
    # TODO: cast numeric fields when appropriate

    document = {}
    return document


## Step 6 - Create the ingestion function

The ingestion function should:

1. read one CSV file
2. optionally limit the number of rows
3. map each row to a document
4. insert the documents into `student_performance_raw`
5. return how many documents were inserted

In [ ]:
def ingest_data(executor, csv_path, limit=None):
    """Read a CSV file and insert its mapped rows into MongoDB."""

    # TODO: load the CSV with pandas
    # TODO: apply the optional limit
    # TODO: convert rows to mapped documents
    # TODO: insert documents into MongoDB
    # TODO: return the number of inserted documents

    return 0


## Step 7 - Small validation checks

Do not perform EDA yet.

Only validate that the pipeline is working. Examples:

- total documents in the new collection
- total documents by source file
- one sample document after ingestion
- one manual insert/update/delete cycle if needed

In [ ]:
# TODO: ingest a small batch first, for example 5 or 10 rows
# TODO: validate the collection count
# TODO: inspect one sample document
# TODO: if everything looks good, ingest the full file(s)


## Suggested class tasks

1. Finish the `MongoDatabase` class.
2. Finish the `MongoExecutor` class.
3. Connect to `university_mongo`.
4. Create and use the new collection `student_performance_raw`.
5. Load `student-mat.csv`.
6. Implement `map_external_record()`.
7. Implement `ingest_data()`.
8. Validate counts and inspect one ingested document.
9. Repeat for `student-por.csv` if time allows.

This notebook should end with a working ingestion pipeline, not with EDA.